# ЗПАД | Лабораторна робота № 2
### Тема: Наука про дані: підготовчий етап
**Частина:** 1
---
**Виконала:** Городиська Анна ФБ-44

### 1: Імпорт бібліотек.
Спочатку підключаємо все необхідне.

In [7]:
import urllib.request
import urllib.error
import os
from datetime import datetime
import pandas as pd

### 2: Завантаження даних. 
Завдання: Завантажити файли з VHI-індексом для кожної області, додати дату та час, передбачити запобігання повторного завантаження.

Для виконання цього завдання я скачала 27 областей (включаючи Крим та міста зі спецстатусом). Виключила також нульовий ID (всю Україну).

In [8]:
def download_vhi_data(folder_path="data"):
    # Спочатку перевіряємо, чи існує папка, якщо ні - створюємо (це зроблено щоб не створювати купу однакових папок)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    for province_id in range(1, 28):
        # Шукаємо, чи вже є файл для цієї області в папці
        existing_files = [f for f in os.listdir(folder_path) if f.startswith(f"vhi_id_{province_id}_")]
        
        if existing_files:
            print(f"Дані для області {province_id} вже існують: {existing_files[0]}. Пропускаємо...")
            continue
            
        # Формуємо URL та ім'я файлу
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        filename = f"vhi_id_{province_id}_{now}.csv"
        filepath = os.path.join(folder_path, filename)
        
        try:
            # Завантажуємо файл
            with urllib.request.urlopen(url) as response:
                html = response.read()
                
            with open(filepath, 'wb') as f:
                f.write(html)
            print(f"Успішно завантажено: {filename}")
            
        except urllib.error.URLError as e:
            print(f"Помилка завантаження області {province_id}: {e}")
            
download_vhi_data()

Дані для області 1 вже існують: vhi_id_1_2026-03-06_20-53-07.csv. Пропускаємо...
Дані для області 2 вже існують: vhi_id_2_2026-03-06_20-53-13.csv. Пропускаємо...
Дані для області 3 вже існують: vhi_id_3_2026-03-06_20-53-14.csv. Пропускаємо...
Дані для області 4 вже існують: vhi_id_4_2026-03-06_20-53-16.csv. Пропускаємо...
Дані для області 5 вже існують: vhi_id_5_2026-03-06_20-53-18.csv. Пропускаємо...
Дані для області 6 вже існують: vhi_id_6_2026-03-06_20-53-19.csv. Пропускаємо...
Дані для області 7 вже існують: vhi_id_7_2026-03-06_20-53-20.csv. Пропускаємо...
Дані для області 8 вже існують: vhi_id_8_2026-03-06_20-53-21.csv. Пропускаємо...
Дані для області 9 вже існують: vhi_id_9_2026-03-06_20-53-22.csv. Пропускаємо...
Дані для області 10 вже існують: vhi_id_10_2026-03-06_20-53-23.csv. Пропускаємо...
Дані для області 11 вже існують: vhi_id_11_2026-03-06_20-53-24.csv. Пропускаємо...
Дані для області 12 вже існують: vhi_id_12_2026-03-06_20-53-25.csv. Пропускаємо...
Дані для області 13 вж

### 3: Очищення та індексація (Data Cleaning)
Завдання: Зчитати файли у фрейм, прибрати зайві дані, змінити індекси областей на українську абетку.

In [10]:
def prepare_dataframe(folder_path="data"):
    # Словник для заміни індексів NOAA на українські 
    noaa_to_ua = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
        11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
        20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
    }
    
    ua_names = {
        1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька',
        5: 'Житомирська', 6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська',
        9: 'Київська', 10: 'Кіровоградська', 11: 'Луганська', 12: 'Львівська',
        13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська', 16: 'Рівненська',
        17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
        21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська',
        25: 'Республіка Крим', 26: 'Київ (місто)', 27: 'Севастополь (місто)'
    }
    
    all_df = []
    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
    
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            filepath = os.path.join(folder_path, filename)
            
            noaa_id = int(filename.split('_')[2])
            ua_id = noaa_to_ua[noaa_id]
            
            # Спочатку зчитуємо файл
            df = pd.read_csv(filepath, header=1, names=headers)
            
            # Видаляємо пустий стовпець, якщо він з'явився звичайно
            if 'empty' in df.columns:
                df = df.drop(columns=['empty'])
                
            # Тут починається дата клінінг
            
            # Перетворюємо стовпець Year на текст і вирізаємо HTML-сміття
            df['Year'] = df['Year'].astype(str).str.replace('<tt><pre>', '').str.replace('</pre></tt>', '').str.strip()
            
            # Залишаємо тільки ті рядки, де в стовпці Year залишились виключно цифри 
            df = df[df['Year'].str.isnumeric()]
            
            # Тут ми конвертуємо типи даних у числові
            df['Year'] = df['Year'].astype(int)
            df['Week'] = df['Week'].astype(int)
            
            # Для VHI використовуємо pd.to_numeric ( частина errors='coerce' перетворить помилки на NaN)
            df['VHI'] = pd.to_numeric(df['VHI'], errors='coerce')
            
            # Видаляємо рядки з відсутніми даними (NaN) та з VHI <= 0 (бо -1 це маркер відсутності даних)
            df = df.dropna()
            df = df[df['VHI'] > 0]
            
            # Додаємо правильні індекси та назви
            df['Area_ID'] = ua_id
            df['Area_Name'] = ua_names[ua_id]
            
            all_df.append(df)
            
    # Зліплюємо все в одну таблицю
    final_df = pd.concat(all_df, ignore_index=True)
    return final_df

# Викликаємо і дивимось на результат
df = prepare_dataframe()
display(df.head())

,Year,Week,SMN,SMT,VCI,TCI,VHI,Area_ID,Area_Name
0,1982,1,0.059,258.24,51.11,48.78,49.95,21,Хмельницька
1,1982,2,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
2,1982,3,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
3,1982,4,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
4,1982,5,0.058,266.42,46.87,28.57,37.72,21,Хмельницька


### 4: Завдання

4.1: Реалізувати процедури для формування вибірок (ряд VHI за рік)

In [17]:
def get_vhi_series(df, province_id, year):
    # Фільтруємо дані за ID області та роком 
    result = df[(df['Area_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    return result

print("--- 1. VHI для 1-ї області (це Вінницька область) за 2020 рік ---")
display(get_vhi_series(df, province_id=1, year=2020).head())

--- 1. VHI для 1-ї області (це Вінницька область) за 2020 рік ---


,Week,VHI
34716,1,40.92
34717,2,43.19
34718,3,44.74
34719,4,45.29
34720,5,44.80


4.2: Реалізувати процедури для формування вибірок (ряд VHI  за діапазон).

In [13]:
def get_vhi_for_years_and_areas(dataframe, area_ids_list, year_start, year_end):
    result = dataframe[
        (dataframe['Area_ID'].isin(area_ids_list)) & 
        (dataframe['Year'] >= year_start) & 
        (dataframe['Year'] <= year_end)
    ]
    return result[['Year', 'Week', 'Area_Name', 'VHI']]
    print("\n--- 2. VHI для Київської (9) та Одеської (14) за 2015-2017 ---")
display(get_vhi_for_years_and_areas(df, [9, 14], 2015, 2017).head())

,Year,Week,Area_Name,VHI
3852,2015,1,Київська,46.27
3853,2015,2,Київська,47.79
3854,2015,3,Київська,48.26
3855,2015,4,Київська,47.77
3856,2015,5,Київська,46.46


4.3: Реалізувати процедури для формування вибірок (екстремуми).

In [16]:
def get_vhi_extremes(dataframe, area_ids_list, year_start, year_end):
    filtered = dataframe[
        (dataframe['Area_ID'].isin(area_ids_list)) & 
        (dataframe['Year'] >= year_start) & 
        (dataframe['Year'] <= year_end)
    ]
    stats = filtered.groupby('Area_Name')['VHI'].agg(['min', 'max', 'mean', 'median']).reset_index()
    return stats
    print("\n--- 3. Екстремуми для Львівської (12) та Харківської (19) за 2010-2020 ---")
display(get_vhi_extremes(df, [12, 19], 2010, 2020))

,Area_Name,min,max,mean,median
0,Львівська,32.39,67.60,49.661678,49.530
1,Харківська,20.89,74.95,46.156801,46.055
